In [2]:
# 01. ESTATÍSTICA INFERENCIAL PARA BIOMARCADORES DE DESEQUILÍBRIO E/I

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import os

print("\n" + "="*80)
print("Iniciando Análise Estatística Inferencial (Congresso FALAN 2026)")
print("="*80)

# 1. Carregamento e Restauração dos Dados
DIR_REPORTS = '../reports/'
df_frames = pd.read_csv(os.path.join(DIR_REPORTS, 'features_frame_a_frame_ERSP.csv'))

# Isolando a tarefa alvo e recriando a matriz por paciente (média das épocas)
df_feliz = df_frames[df_frames['Condicao'] == 'Face Feliz'].copy()
colunas_agrupamento = ['ID', 'Grupo']
features_numericas = [c for c in df_feliz.columns if c not in ['ID', 'Grupo', 'Condicao', 'Tipo', 'Frame_Num']]
df_agrupado = df_feliz.groupby(colunas_agrupamento)[features_numericas].mean().reset_index()

# 2. Definição dos Marcadores de E/I a serem testados
marcadores_ei = [
    'AlphaRel_Fp2',  # Proxy de inibição ativa
    'GammaRel_T4',   # Proxy de excitação interneuronal local
    'TBR_Fp2',       # Razão de tônus (Inibição de base vs Ativação)
    'Entropia_Fp2'   # Proxy de ruído do sistema (Desregulação)
]

# Configuração visual para publicação
sns.set_theme(style="whitegrid", palette="muted")
cores_grupos = {'Control': '#2ca02c', 'TEA': '#d62728'} # Verde e Vermelho

resultados_estatisticos = []

# 3. O Pipeline Matemático
for marcador in marcadores_ei:
    if marcador not in df_agrupado.columns:
        print(f"Atenção: A variável {marcador} não foi encontrada na matriz. Pulando...")
        continue
        
    dados_controle = df_agrupado[df_agrupado['Grupo'] == 'Control'][marcador].values
    dados_tea = df_agrupado[df_agrupado['Grupo'] == 'TEA'][marcador].values
    
    # A. Teste de Normalidade (Shapiro-Wilk)
    _, p_shapiro_c = stats.shapiro(dados_controle)
    _, p_shapiro_t = stats.shapiro(dados_tea)
    
    normal = (p_shapiro_c > 0.05) and (p_shapiro_t > 0.05)
    
    # B. Escolha do Teste e Cálculo do Tamanho do Efeito
    if normal:
        nome_teste = "Teste T Independente"
        estatistica, p_valor = stats.ttest_ind(dados_controle, dados_tea, equal_var=False)
        
        # Cohen's d para paramétricos
        nx, ny = len(dados_controle), len(dados_tea)
        dof = nx + ny - 2
        pool_sd = np.sqrt(((nx-1)*np.var(dados_controle, ddof=1) + (ny-1)*np.var(dados_tea, ddof=1)) / dof)
        tamanho_efeito = (np.mean(dados_controle) - np.mean(dados_tea)) / pool_sd
        nome_efeito = "Cohen's d"
        
    else:
        nome_teste = "Mann-Whitney U"
        estatistica, p_valor = stats.mannwhitneyu(dados_controle, dados_tea, alternative='two-sided')
        
        # Rank-Biserial Correlation para não-paramétricos
        nx, ny = len(dados_controle), len(dados_tea)
        u1 = estatistica
        u2 = nx * ny - u1
        tamanho_efeito = 1 - (2 * min(u1, u2)) / (nx * ny)
        nome_efeito = "Rank-Biserial (r)"

    # C. Estruturação do Log
    direcao = "Nenhuma"
    if p_valor < 0.05:
        if np.median(dados_tea) > np.median(dados_controle):
            direcao = "TEA apresenta valores maiores (Aumento)"
        else:
            direcao = "TEA apresenta valores menores (Redução)"
            
    resultados_estatisticos.append({
        'Marcador': marcador,
        'Distribuição': 'Normal' if normal else 'Não-Normal',
        'Teste Realizado': nome_teste,
        'P-Valor': p_valor,
        'Significativo': 'SIM' if p_valor < 0.05 else 'NÃO',
        'Direção': direcao,
        f'{nome_efeito}': abs(tamanho_efeito)
    })

    # D. Plotagem do Gráfico (Violin + Swarmplot para ver cada paciente)
    plt.figure(figsize=(8, 6))
    ax = sns.violinplot(x='Grupo', y=marcador, data=df_agrupado, palette=cores_grupos, inner="box", alpha=0.5)
    sns.stripplot(x='Grupo', y=marcador, data=df_agrupado, color='black', alpha=0.6, jitter=True, size=6)
    
    # Inserindo estrelas de significância no gráfico
    sig_symbol = 'ns'
    if p_valor < 0.001: sig_symbol = '***'
    elif p_valor < 0.01: sig_symbol = '**'
    elif p_valor < 0.05: sig_symbol = '*'
        
    y_max = df_agrupado[marcador].max()
    offset = (y_max - df_agrupado[marcador].min()) * 0.05
    plt.plot([0, 0, 1, 1], [y_max + offset, y_max + offset*1.5, y_max + offset*1.5, y_max + offset], lw=1.5, color='black')
    plt.text(0.5, y_max + offset*1.5, sig_symbol, ha='center', va='bottom', color='black', fontsize=14, fontweight='bold')
    
    plt.title(f'Distribuição de {marcador} durante Processamento de Faces\n(p = {p_valor:.4f})', fontsize=14, pad=15)
    plt.ylabel(f'{marcador} (Unidades Arbitrárias)', fontsize=12)
    plt.xlabel('')
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_REPORTS, f'plot_inferencia_{marcador}.png'), dpi=300)
    plt.close()

# 4. Exibição do Laudo Estatístico
df_resultados = pd.DataFrame(resultados_estatisticos)
print("\n[LAUDO ESTATÍSTICO INFERENCIAL]")
print(df_resultados.to_string(index=False))
print("\n" + "="*80)
print(f"Gráficos de violino exportados com sucesso para a pasta '{DIR_REPORTS}'.")


Iniciando Análise Estatística Inferencial (Congresso FALAN 2026)


/tmp/ipykernel_7963/452661447.py:96: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.violinplot(x='Grupo', y=marcador, data=df_agrupado, palette=cores_grupos, inner="box", alpha=0.5)
/tmp/ipykernel_7963/452661447.py:96: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.violinplot(x='Grupo', y=marcador, data=df_agrupado, palette=cores_grupos, inner="box", alpha=0.5)
/tmp/ipykernel_7963/452661447.py:96: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.violinplot(x='Grupo', y=marcador, data=df_agrupado, palette=cores_grupos, inner="box", alpha=0.5)
/tmp/ipykerne


[LAUDO ESTATÍSTICO INFERENCIAL]
    Marcador Distribuição      Teste Realizado  P-Valor Significativo                                 Direção  Cohen's d  Rank-Biserial (r)
AlphaRel_Fp2       Normal Teste T Independente 0.000900           SIM TEA apresenta valores menores (Redução)   1.094478                NaN
 GammaRel_T4   Não-Normal       Mann-Whitney U 0.000997           SIM TEA apresenta valores maiores (Aumento)        NaN           0.601852
     TBR_Fp2   Não-Normal       Mann-Whitney U 0.130463           NÃO                                 Nenhuma        NaN           0.277778
Entropia_Fp2       Normal Teste T Independente 0.261276           NÃO                                 Nenhuma   0.402586                NaN

Gráficos de violino exportados com sucesso para a pasta '../reports/'.
